# TIGER SemanticID: Qwen3-8B Fine-tuning for SID Recommendation (IMPROVED)

This notebook fine-tunes Qwen3-8B to generate Semantic IDs for next-item recommendation using **multi-type training data** and **improved hyperparameters**.

## Key Improvements

**Data Generation (2.1x increase):**
- Type A: SID → Title (~73K examples)
- Type B: Title → SID with augmentation (~109K examples)
- Type C: Next-item prediction (~416K examples)
- Type D: Semantic understanding (~5K examples)
- Type E: Co-occurrence patterns (~257K examples)
- **Total: ~870K examples** (was 416K with Type C only)

**Training Improvements:**
- Stage A epochs: 1 → 3 (better embedding learning for 1,024 SID tokens)
- Stage B learning rate: 1e-4 → 2e-5 (prevent mode collapse)
- LoRA rank: **r=16** (matched to data availability, ~36M trainable params)
- LoRA alpha: 32 (2× rank)
- Logging: Reduced to ~11 logs per epoch for cleaner output

**Data-to-Parameter Analysis:**
- Stage A: 870K / 8.4M = 103 examples per param ✓ Excellent
- Stage B: 870K / 36M = 0.024 examples per param (41 params per example)
- With LoRA regularization + pre-trained base, this is workable
- Reference also has low ratio (4.2M / 71M = 0.059 examples per param)

**Remaining Issues (TODO):**
- Loss masking still trains on full conversation (needs code fix)
- No training constraints (constraints only at inference)
- Teacher forcing vs sequential generation mismatch

## Pipeline Overview

1. **Prepare Metadata**: Extract product titles for Types A & B
2. **Build Dialogs**: Generate multi-type conversational data + trie
3. **Stage A (Vocab)**: Fine-tune only embeddings (3 epochs)
4. **Stage B (LoRA)**: Fine-tune with LoRA adapters (improved params)
5. **Inference**: Generate SIDs with level + trie constraints
6. **Evaluation**: Measure SID@K, Invalid-ID@K, diversity

## Expected Improvements

With these fixes, we expect:
- Invalid-ID@1: 100% → <10% (was complete failure)
- Unique SIDs: 8 → 1000+ (was mode collapse)
- SID@1: 0% → 5-15% (was zero accuracy)
- More stable training with lower LR and better data-to-param ratio

## Setup

In [1]:
# Install dependencies (Colab)
!pip install -q transformers accelerate peft datasets bitsandbytes tiktoken sentencepiece jsonlines orjson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 20.1 MB/s eta 0:00:00


In [2]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
assert os.path.exists('/content/drive')
WORK_DIR = '/content/drive/MyDrive/colab/tiger_semantic_id'
%mkdir -p $WORK_DIR
%cd $WORK_DIR

/content/drive/MyDrive/colab/tiger_semantic_id


In [4]:
# Create LLM building folder
import shutil
import os

LLM_DIR = WORK_DIR + '/llm_finetuning'

# Remove existing folder if it exists
if os.path.exists(LLM_DIR):
    print(f"Removing existing {LLM_DIR} folder...")
    shutil.rmtree(LLM_DIR)

# Create fresh folder
os.makedirs(LLM_DIR)
print(f"✅ Created fresh {LLM_DIR} folder")

✅ Created fresh /content/drive/MyDrive/colab/tiger_semantic_id/llm_finetuning folder


In [5]:
# Specify data preparation artifacts location
RQVAE_DIR = WORK_DIR + '/rq_vae_building/artifacts'

print("=" * 60)
print("DATA DEPENDENCY CONFIGURATION")
print("=" * 60)
print(f"\nData source: {RQVAE_DIR}")
print(f"Exists: {os.path.exists(RQVAE_DIR)}")

if not os.path.exists(RQVAE_DIR):
    print("\n⚠️  Data preparation artifacts not found!")
    print("   Please run TIGER_SemanticID.ipynb first")
else:
    print("\n✅ Data preparation artifacts found")
    print("\nAvailable files:")
    for f in sorted(os.listdir(RQVAE_DIR)):
        filepath = os.path.join(RQVAE_DIR, f)
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f"  - {f} ({size_mb:.2f} MB)")

print("=" * 60)

DATA DEPENDENCY CONFIGURATION

Data source: /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building/artifacts
Exists: True

✅ Data preparation artifacts found

Available files:
  - figs_c1_category.png (0.06 MB)
  - figs_hierarchy.png (0.11 MB)
  - item_metadata.json (12.21 MB)
  - item_to_sid.json (2.16 MB)
  - prefix_to_items.json (1.46 MB)
  - rqvae.pt (1.89 MB)
  - semantic_ids.npy (0.62 MB)
  - seq2seq.pt (14.33 MB)
  - sid_to_items.json (1.93 MB)
  - user_sequences.json (7.58 MB)


In [6]:
# Clone repo, install dependencies, and make src importable (Colab-friendly)
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = '20250908_tiger_dev'

import os
if IN_COLAB:
    if os.path.exists(repo_dir):
      !rm -rf {repo_dir}
    !git clone $repo_url
    %cd $repo_dir
    !git fetch --all
    !git checkout $branch_name || echo 'Branch not found; staying on default.'


Cloning into 'recsys_playground'...
remote: Enumerating objects: 707, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 707 (delta 187), reused 135 (delta 107), pack-reused 473 (from 1)
Receiving objects: 100% (707/707), 315.82 KiB | 1.77 MiB/s, done.
Resolving deltas: 100% (486/486), done.
/content/drive/MyDrive/colab/tiger_semantic_id/recsys_playground
Fetching origin
Branch '20250908_tiger_dev' set up to track remote branch '20250908_tiger_dev' from 'origin'.
Switched to a new branch '20250908_tiger_dev'


In [8]:
# Imports
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, f'{WORK_DIR}/recysys_playground/tiger_semantic_id/src')

# Config
ARTIFACTS_DIR = RQVAE_DIR

print(f"Artifacts: {ARTIFACTS_DIR}")
print(f"LLM outputs: {LLM_DIR}")

Artifacts: /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building/artifacts
LLM outputs: /content/drive/MyDrive/colab/tiger_semantic_id/llm_finetuning


## 1. Prepare Metadata & Build Multi-Type Dialogs

**Step 1a: Prepare Metadata (Required for Types A & B)**
- Extract product titles from meta_Beauty.json.gz
- Create mapping: item_id → {'title': '...'}
- Enables SID↔Title bidirectional tasks

**Step 1b: Generate Multi-Type Training Data**
- Type A: SID → Title (per-item, teaches semantic meaning)
- Type B: Title → SID (reverse mapping, 3x augmentation)
- Type C: Next-item prediction (sequential patterns)
- Type D: Semantic understanding (hierarchical relationships)
- Type E: Co-occurrence patterns (associative relationships)

**Data Type Selection:**
- Default: All types "A,B,C,D,E" (~870K examples)
- Fallback: Type "C,D,E" if metadata missing (~680K examples)
- Minimum: Type "C" only (~416K examples)

In [9]:
# Build dialogs for all data types
# Options:
#   --data_types: Comma-separated list (A,B,C,D,E)
#     A: SID → Title (requires metadata)
#     B: Title → SID (requires metadata, with prompt augmentation)
#     C: Next-item prediction (sequential recommendation)
#     D: Semantic understanding (SID relationship questions)
#     E: Co-purchase patterns (collaborative filtering signals)

# Default: Only Type C (next-item prediction)
# DATA_TYPES = "C"

# With all types (requires metadata):
DATA_TYPES = "A,B,C,D,E"

# Build command based on available metadata
metadata_path = f'{ARTIFACTS_DIR}/item_metadata.json'
has_metadata = os.path.exists(metadata_path)

if not has_metadata and any(t in DATA_TYPES for t in ['A', 'B']):
    print("Warning: Types A & B require metadata, but metadata file not found.")
    print("Falling back to Type C only.")
    DATA_TYPES = "C"

# Build command
cmd = f"""python -m tiger_semantic_id.src.llm.build_sid_dialogs \\
    --artifacts_dir {ARTIFACTS_DIR} \\
    --out {LLM_DIR} \\
    --data_types "{DATA_TYPES}" \\
    --history_lengths "2,3,5" \\
    --train_ratio 0.95"""

# Add metadata path if available
if has_metadata:
    cmd += f""" \\
    --metadata_path {metadata_path}"""

# Add optional parameters for Type D & E
cmd += """ \\
    --semantic_sample_size 5000 \\
    --copurchase_top_k 10 \\
    --copurchase_examples_per_item 3"""

print(f"Generating data types: {DATA_TYPES}\n")
!{cmd}

Generating data types: A,B,C,D,E

Generating data types: A, B, C, D, E

Loading artifacts from /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building/artifacts...
Loaded 80844 semantic IDs

Building SID trie...
Saved trie to /content/drive/MyDrive/colab/tiger_semantic_id/llm_finetuning/sid_trie.pkl
  - valid_c2: 256 L1 codes
  - valid_c3: 18225 (L1,L2) pairs
  - valid_c4: 60543 (L1,L2,L3) triples

Loading item metadata from /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building/artifacts/item_metadata.json...
Loaded metadata for 80840 items

Loading user sequences from /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building/artifacts/user_sequences.json...
Loaded sequences for 117742 users
Average sequence length: 8.14

GENERATING TRAINING DIALOGS

[Type A] SID → Title
Building SID→Title dialogs: 100% 80844/80844 [00:00<00:00, 172770.37it/s]
  Generated 80,840 examples

[Type B] Title → SID
Building Title→SID dialogs: 100% 80844/80844 [00:01<00:00, 69734.41it/s]


In [10]:
# Analyze generated data
import jsonlines
from collections import Counter

train_path = f'{LLM_DIR}/dialogs_train.jsonl'
valid_path = f'{LLM_DIR}/dialogs_valid.jsonl'
trie_path = f'{LLM_DIR}/sid_trie.pkl'

print("="*60)
print("GENERATED DATA ANALYSIS")
print("="*60)

# Load dialogs
with jsonlines.open(train_path) as reader:
    train_dialogs = list(reader)

with jsonlines.open(valid_path) as reader:
    valid_dialogs = list(reader)

total_dialogs = len(train_dialogs) + len(valid_dialogs)
print(f"\n📊 Total dialogs: {total_dialogs:,}")
print(f"  Train: {len(train_dialogs):,} ({len(train_dialogs)/total_dialogs*100:.1f}%)")
print(f"  Valid: {len(valid_dialogs):,} ({len(valid_dialogs)/total_dialogs*100:.1f}%)")
print(f"  Trie exists: {'✓' if os.path.exists(trie_path) else '✗'}")

# Count by type
type_counts = Counter(d.get('type', 'unknown') for d in train_dialogs + valid_dialogs)

print(f"\n📈 Breakdown by data type:")
for dtype, count in sorted(type_counts.items()):
    pct = count / total_dialogs * 100
    print(f"  {dtype:30s}: {count:7,} ({pct:5.2f}%)")

# Show examples from each type
print(f"\n📝 Sample dialogs by type:")
print("="*60)

samples_by_type = {}
for dialog in (train_dialogs + valid_dialogs)[:5000]:  # Sample from first 5K
    dtype = dialog.get('type', 'unknown')
    if dtype not in samples_by_type:
        samples_by_type[dtype] = dialog

for dtype in sorted(samples_by_type.keys()):
    dialog = samples_by_type[dtype]
    print(f"\n[{dtype}]")
    for msg in dialog['messages']:
        content = msg['content']
        if len(content) > 150:
            content = content[:150] + "..."
        print(f"  {msg['role'].upper()}: {content}")

print("\n" + "="*60)

GENERATED DATA ANALYSIS

📊 Total dialogs: 2,741,224
  Train: 2,604,162 (95.0%)
  Valid: 137,062 (5.0%)
  Trie exists: ✓

📈 Breakdown by data type:
  copurchase                    : 242,424 ( 8.84%)
  semantic_understanding        :   5,000 ( 0.18%)
  seq_last_2                    : 723,480 (26.39%)
  seq_last_3                    : 723,480 (26.39%)
  seq_last_5                    : 723,480 (26.39%)
  sid_to_title                  :  80,840 ( 2.95%)
  title_to_sid                  : 242,520 ( 8.85%)

📝 Sample dialogs by type:

[copurchase]
  SYSTEM: You are a recommender that must reply ONLY with the next product's Semantic ID as 4 tokens in order: L1, L2, L3, L4.
Valid token ranges by level:
- L1...
  USER: Users who bought <sid_42> <sid_479> <sid_580> <sid_768> also frequently bought:
  ASSISTANT: <sid_52> <sid_479> <sid_708> <sid_769>

[semantic_understanding]
  SYSTEM: You are a recommender that must reply ONLY with the next product's Semantic ID as 4 tokens in order: L1, L2, L3, L4

## 2. Tokenizer Resize

Add 1,027 new tokens to Qwen tokenizer and initialize embeddings.

In [ ]:
# Resize tokenizer and model
!python -m tiger_semantic_id.src.llm.tokenizer_resize_qwen \
    --base Qwen/Qwen2.5-8B-Instruct \
    --out $LLM_DIR/qwen3_vocab_stage \
    --torch_dtype bfloat16

## 3. Stage A: Vocabulary Extension

Fine-tune **only embeddings** to teach the model the new SID tokens.

In [ ]:
# Stage A: Embeddings only
!python -m tiger_semantic_id.src.llm.finetune_qwen_vocab \
    --data $LLM_DIR/dialogs_train.jsonl \
    --valid $LLM_DIR/dialogs_valid.jsonl \
    --in_model $LLM_DIR/qwen3_vocab_stage \
    --out_model $LLM_DIR/qwen3_vocab_stage \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --learning_rate 5e-4 \
    --num_train_epochs 1 \
    --warmup_ratio 0.03 \
    --logging_steps 50 \
    --save_steps 500 \
    --bf16 \
    --gradient_checkpointing

In [ ]:
# Stage A: Embeddings only (IMPROVED PARAMETERS)
# Critical fix: Increased epochs from 1 → 3 for better embedding learning
# 1,024 new SID tokens need sufficient training to learn meaningful representations

!python -m tiger_semantic_id.src.llm.finetune_qwen_vocab \
    --data $LLM_DIR/dialogs_train.jsonl \
    --valid $LLM_DIR/dialogs_valid.jsonl \
    --in_model $LLM_DIR/qwen3_vocab_stage \
    --out_model $LLM_DIR/qwen3_vocab_stage \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --learning_rate 5e-4 \
    --num_train_epochs 3 \
    --warmup_ratio 0.03 \
    --logging_steps 2500 \
    --save_steps 5000 \
    --bf16 \
    --gradient_checkpointing

print("\n✓ Stage A complete - SID token embeddings trained for 3 epochs")

## 4. Stage B: LoRA Fine-tuning

Fine-tune with **LoRA adapters** for memory-efficient training (~20-25GB VRAM instead of ~60GB).

In [ ]:
# Stage B: LoRA fine-tuning with memory optimizations
!python -m tiger_semantic_id.src.llm.finetune_qwen_lora \
    --data $LLM_DIR/dialogs_train.jsonl \
    --valid $LLM_DIR/dialogs_valid.jsonl \
    --in_model $LLM_DIR/qwen3_vocab_stage \
    --out_model $LLM_DIR/qwen3_lora_adapter \
    --sid_trie $LLM_DIR/sid_trie.pkl \
    --lora_r 16 \
    --lora_alpha 32 \
    --lora_dropout 0.05 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --learning_rate 1e-4 \
    --num_train_epochs 3 \
    --warmup_ratio 0.03 \
    --logging_steps 50 \
    --save_steps 500 \
    --bf16 \
    --gradient_checkpointing

In [ ]:
# Stage B: LoRA fine-tuning (IMPROVED PARAMETERS)
# Critical fixes applied:
# 1. Learning rate: 1e-4 → 2e-5 (5x reduction to prevent mode collapse)
# 2. LoRA rank: 16 (REDUCED from 32 to match data availability)
#    - 870K examples / 36M params = 24 params per example
#    - With r=32: 81 params/example (severe underfitting risk)
#    - With r=16: 41 params/example (better data efficiency)
# 3. Training on expanded multi-type dataset (~870K examples)
#
# Data-to-parameter analysis:
# - Stage B trainable params: ~36M (with r=16)
# - 870K examples = 0.024 examples per param (low but workable with LoRA)
# - LoRA's low-rank constraint + pre-trained base makes this viable
#
# Note: Loss masking issue still exists in finetune_qwen_lora.py
# TODO: Update training code to only compute loss on assistant tokens

!python -m tiger_semantic_id.src.llm.finetune_qwen_lora \
    --data $LLM_DIR/dialogs_train.jsonl \
    --valid $LLM_DIR/dialogs_valid.jsonl \
    --in_model $LLM_DIR/qwen3_vocab_stage \
    --out_model $LLM_DIR/qwen3_lora_adapter \
    --sid_trie $LLM_DIR/sid_trie.pkl \
    --lora_r 16 \
    --lora_alpha 32 \
    --lora_dropout 0.05 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --learning_rate 2e-5 \
    --num_train_epochs 3 \
    --warmup_ratio 0.03 \
    --logging_steps 2500 \
    --save_steps 5000 \
    --bf16 \
    --gradient_checkpointing

print("\n✓ Stage B complete - LoRA adapter trained with improved hyperparameters")

In [ ]:
# Interactive inference with LoRA adapter
from tiger_semantic_id.src.llm.inference_qwen import SIDRecommender
import json

# Load recommender with LoRA adapter
recommender = SIDRecommender(
    model_path=f'{LLM_DIR}/qwen3_lora_adapter',
    base_model_path=f'{LLM_DIR}/qwen3_vocab_stage',
    trie_path=f'{LLM_DIR}/sid_trie.pkl',
    is_lora_adapter=True,
)

# Load mappings
with open(f'{ARTIFACTS_DIR}/sid_to_items.json') as f:
    sid_to_items = json.load(f)

print("Recommender loaded!")

In [ ]:
# Example: Generate from history
history_sids = [
    (64, 54, 125, 0),
    (64, 156, 194, 0),
    (112, 191, 11, 4),
]

results = recommender.recommend(
    history_sids=history_sids,
    sid_to_items=sid_to_items,
    top_k=5,
)

print("\n=== Generated Recommendation ===")
if results:
    result = results[0]
    print(f"Generated SID: {result['sid']}")
    print(f"Mapped Items:")
    for item_id in result['items']:
        print(f"  - {item_id}")
else:
    print("No valid SID generated")

## 6. Evaluation

Measure SID@K, Invalid-ID rate, and qualitative examples.

In [ ]:
# Evaluate on validation set
import numpy as np
from tqdm import tqdm

# Load validation dialogs
eval_size = min(1000, len(valid_dialogs))
eval_dialogs = valid_dialogs[:eval_size]

print(f"Evaluating on {eval_size} examples...")

# Metrics
invalid_count = 0
sid_hits = []
generated_sids = []

for dialog in tqdm(eval_dialogs):
    # Extract ground truth
    assistant_msg = dialog['messages'][2]['content']
    # Parse SID tokens from assistant message
    tokens = assistant_msg.split()
    if len(tokens) != 4:
        continue

    # Extract codes from tokens
    try:
        gt_codes = []
        for i, token in enumerate(tokens):
            code_num = int(token.split('_')[1].rstrip('>'))
            level_offset = i * 256
            code = code_num - level_offset
            gt_codes.append(code)
        gt_sid = tuple(gt_codes)
    except:
        continue

    # Extract history from user message (NEW COMPACT FORMAT)
    user_msg = dialog['messages'][1]['content']
    history_sids = []

    # New format: "User's last purchases: <sid_X> <sid_Y> <sid_Z> <sid_W>, <sid_A> <sid_B> <sid_C> <sid_D>. Next:"
    if "User's last purchases:" in user_msg:
        try:
            # Extract the history part between "User's last purchases:" and ". Next:"
            history_part = user_msg.split("User's last purchases:")[1].split(". Next:")[0].strip()

            # Split by comma to get individual items
            history_items = [item.strip() for item in history_part.split(",")]

            for item in history_items:
                tokens = item.split()
                if len(tokens) != 4:
                    continue

                codes = []
                for i, token in enumerate(tokens):
                    code_num = int(token.split('_')[1].rstrip('>'))
                    level_offset = i * 256
                    code = code_num - level_offset
                    codes.append(code)
                history_sids.append(tuple(codes))
        except Exception as e:
            # Fallback to old format if parsing fails
            history_lines = user_msg.split('\n')[1:-1]  # Skip "History:" and "Recommend next:"
            for line in history_lines:
                tokens = line.split()
                if len(tokens) != 4:
                    continue
                try:
                    codes = []
                    for i, token in enumerate(tokens):
                        code_num = int(token.split('_')[1].rstrip('>'))
                        level_offset = i * 256
                        code = code_num - level_offset
                        codes.append(code)
                    history_sids.append(tuple(codes))
                except:
                    continue

    if not history_sids:
        continue

    # Generate SID
    try:
        generated_sid = recommender.generate_sid(history_sids=history_sids)

        if generated_sid is None:
            invalid_count += 1
            continue

        generated_sids.append(generated_sid)

        # Check if valid (exists in catalog)
        sid_key = ','.join(map(str, generated_sid))
        if sid_key not in sid_to_items:
            invalid_count += 1
            continue

        # Check if matches ground truth
        sid_hits.append(1 if generated_sid == gt_sid else 0)

    except Exception as e:
        print(f"Error: {e}")
        invalid_count += 1
        continue

print("\n=== Evaluation Results ===")
print(f"Examples evaluated: {len(sid_hits) + invalid_count}")
print(f"Invalid-ID@1: {invalid_count / (len(sid_hits) + invalid_count) * 100:.2f}%")
print(f"SID@1 (exact match): {np.mean(sid_hits) * 100:.2f}%" if sid_hits else "N/A")
print(f"Unique SIDs generated: {len(set(generated_sids))}")

In [ ]:
# Qualitative examples
print("\n=== Qualitative Examples ===")

test_histories = [
    [(91, 54, 165, 0), (146, 204, 254, 0), (225, 239, 96, 0)],
    [(229, 236, 102, 0), (225, 212, 226, 1)],
    [(94, 233, 248, 0), (180, 191, 245, 0), (89, 141, 245, 0)],
]

for i, history in enumerate(test_histories, 1):
    print(f"\n[Example {i}]")
    print(f"History: {history}")

    generated_sid = recommender.generate_sid(history_sids=history)
    print(f"Generated SID: {generated_sid}")

    if generated_sid:
        sid_key = ','.join(map(str, generated_sid))
        items = sid_to_items.get(sid_key, [])
        print(f"Mapped to {len(items)} items")
        if items:
            print(f"  Top item: {items[0]}")

## 7. Acceptance Criteria

Check if all acceptance criteria pass:

1. ✅ Stage A completes and new tokens are learned
2. ✅ Stage B completes with validation loss decreasing
3. ✅ Invalid-ID@1 = 0% (or very close)
4. ✅ SID@10 ≥ baseline
5. ✅ NL prompts produce valid SIDs

In [ ]:
# Summary
print("\n" + "="*60)
print("ACCEPTANCE CRITERIA")
print("="*60)

print("\n[1] Stage A: Vocabulary Extension")
vocab_stage_path = f'{LLM_DIR}/qwen3_vocab_stage/pytorch_model.bin'
print(f"  Status: {'✅ PASS' if os.path.exists(vocab_stage_path) else '❌ FAIL'}")

print("\n[2] Stage B: Full Fine-tuning")
full_stage_path = f'{LLM_DIR}/qwen3_full_stage/pytorch_model.bin'
print(f"  Status: {'✅ PASS' if os.path.exists(full_stage_path) else '❌ FAIL'}")

print("\n[3] Invalid-ID Rate")
invalid_rate = invalid_count / (len(sid_hits) + invalid_count) * 100 if (len(sid_hits) + invalid_count) > 0 else 100
print(f"  Invalid-ID@1: {invalid_rate:.2f}%")
print(f"  Status: {'✅ PASS' if invalid_rate < 5.0 else '❌ FAIL'} (target: <5%)")

print("\n[4] SID@1 Exact Match")
sid_acc = np.mean(sid_hits) * 100 if sid_hits else 0
print(f"  SID@1: {sid_acc:.2f}%")
print(f"  Status: {'✅ PASS' if sid_acc > 0 else '❌ FAIL'} (target: >0%)")

print("\n[5] Qualitative Examples")
print(f"  Status: ✅ PASS (see examples above)")

print("\n" + "="*60)
all_pass = (
    os.path.exists(vocab_stage_path) and
    os.path.exists(full_stage_path) and
    invalid_rate < 5.0 and
    sid_acc > 0
)
print(f"OVERALL: {'🎉 ALL CHECKS PASSED!' if all_pass else '❌ SOME CHECKS FAILED'}")
print("="*60)